In [ ]:
# Import libraries
from astropy.io import fits
from astropy.wcs import WCS
import urllib.request
import pdfplumber
import re
import pandas as pd
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
import astropy
from astropy.visualization import ManualInterval
import photutils
from astropy.visualization import AsinhStretch
from astropy.visualization import ZScaleInterval
from photutils.aperture import EllipticalAperture, EllipticalAnnulus
from photutils.centroids import (centroid_1dg, centroid_com)
from photutils.aperture import ApertureStats
from photutils.isophote import Ellipse
from photutils.isophote import EllipseGeometry
from matplotlib.patches import Ellipse as mpl_Ellipse
from astropy.modeling.functional_models import Sersic1D
from astropy.modeling.fitting import LevMarLSQFitter, LMLSQFitter,TRFLSQFitter, DogBoxLSQFitter
from sklearn.preprocessing import StandardScaler, MinMaxScaler
plt.rcParams['figure.dpi'] = 1000

In this notebook I'll be testing out some methods of automating certain steps in the icl measurement process, before compiling it all in an actual script. First thing that comes to mind to automate is finding the mask location of the bcg. The eventual script needs to run sextractor first (in each band gri), and then the python script.

In [ ]:
# Useful functions

# by default we clip and stretch the data before displaying it
def display(mat, vmin, vmax, tag):
    interval = ManualInterval(vmin, vmax)
    mat_1 = interval(mat)
    stretch = AsinhStretch(0.0004)
    mat_2 = stretch(mat_1)

    return mat_2

def display_zscale(mat):
    z1, z2 = ZScaleInterval().get_limits(mat)

    return z1,z2

def get_masked_coadd(band_list):

    masked_coadd_list = []

    for band in band_list:
        hdul_coadd = fits.open(f'{coadd_dir}{cln}_{band}{coadd_fits_filename}')
        hdul_mask = fits.open(f'{mask_dir}{cln}_{band}{mask_fits_filename}')
        data_coadd = hdul_coadd[0].data
        data_mask = hdul_mask[0].data
        masked_coadd = ma.masked_array(data_coadd, mask=data_mask)
        masked_coadd_list.append(masked_coadd)
        # z1, z2 = display_zscale(masked_coadd)
        # plt.figure()
        # plt.imshow(masked_coadd, vmin=z1, vmax=z2)
        # plt.gca().invert_yaxis()
        # plt.title(f'Masked Coadd {band} Zscale Skycorr')
        # plt.savefig(f"masked_coadd_zscale_skycorr_{band}.png", dpi = 4096)

    return masked_coadd_list

def mask_single_coadd(coadd_arr, mask_arr, bcg_val):
    arr = np.where(mask_arr == bcg_val,0,mask_arr)
    arr = np.where(arr != 0,1, arr)
    
    return ma.masked_array(coadd_arr, mask=arr)

def fit_ellip_annuli(band, coadd, center, spacings, theta=45):
    # Array to hold summed intensities in each annulus
    sums = []
    # Array to hold annulus areas
    areas = []

    # Plot coadd
    plt.figure(figsize=(10,8))
    plt.imshow(display(coadd, -0.1, 250, 'fig1')*255)
    plt.plot(center[0],center[1], 'r+')

    # Begin fitting annuli
    for ind in range(len(spacings) - 1):
        annulus = EllipticalAnnulus(center, a_in=spacings[ind], a_out=spacings[ind+1], \
                               b_out=spacings[ind], theta=theta)
        annulus.plot(color='red', lw=1)
        sums.append(list(annulus.do_photometry(coadd)[0]))
        # aperstats = ApertureStats(data, annulus_aperture)
        areas.append(annulus.area)

    # Convert to list
    sums = sum(sums, [])

    # Other plot parameters
    plt.gca().invert_yaxis()
    plt.colorbar(label="Intensity", orientation="vertical") 
    plt.xlabel('x (pixels)')
    plt.ylabel('y (pixels)')
    plt.title(f'{band} Band Elliptical Annuli')
    plt.show()

    # Return annuli sums (units of counts) and areas (untis of pixels)
    return sums, areas
    

In [ ]:
# Directory information

# CLUSTER NAME
cln = 'A85'

mask_dir = '~/data/zescalan_research/A85_testing/SExtractor/'
coadd_dir = '~/data/zescalan_research/A85_testing/'

mask_g_filename = 'seg_g_clips.fits'
mask_r_filename = 'seg_r_clips.fits'
mask_i_filename = 'seg_i_clips.fits'

coadd_filename = '55-66_deepCoadd_clips.fits'

In [ ]:
# Read in files

# Open data
mask_g_hdul = fits.open(mask_dir + mask_g_filename)
mask_r_hdul = fits.open(mask_dir + mask_r_filename)
mask_i_hdul = fits.open(mask_dir + mask_i_filename)

coadd_g_hdul = fits.open(f'{coadd_dir}{cln}_g{coadd_filename}')
coadd_r_hdul = fits.open(f'{coadd_dir}{cln}_r{coadd_filename}')
coadd_i_hdul = fits.open(f'{coadd_dir}{cln}_i{coadd_filename}')

# (8000,8000) arrays
coadd_g_data = coadd_g_hdul[0].data
coadd_r_data = coadd_r_hdul[0].data
coadd_i_data = coadd_i_hdul[0].data

mask_g_data = mask_g_hdul[0].data
mask_r_data = mask_r_hdul[0].data
mask_i_data = mask_i_hdul[0].data

Trying to use table of BCG centers from paper https://arxiv.org/abs/1407.2260 as centers. Need to first retrieve the coords.

In [ ]:
# Testing out creating csv of BCG locations

url = "https://arxiv.org/pdf/1407.2260"

# Download and save
pdf_path = "BCG_coords.pdf"
urllib.request.urlretrieve(url, pdf_path)

target_string = "2000"  # Change this to the string marking the beginning of the table
page_range = range(30, 36)  # Adjust this to the range of pages you want to process

# Create an empty list to store the rows from all pages
all_rows = []

with pdfplumber.open(pdf_path) as pdf:
    # Loop through all pages in the specified range
    for page_number in page_range:
        page = pdf.pages[page_number]
        text = page.extract_text()

        if text:
            # Split text into lines (rows)
            lines = text.split("\n")

            # Find the index of the target string (first occurrence)
            start_index = next((i for i, line in enumerate(lines) if target_string in line), None)

            if start_index is not None:
                # Keep all lines starting from the target string (inclusive)
                filtered_lines = lines[start_index:]

                # Loop through filtered lines and skip lines that don't start with a number
                for line in filtered_lines:
                    # Check if the first non-whitespace character is a digit
                    if re.match(r'^\d', line.strip()):  # Regex checks if line starts with a digit
                        # Convert each line into a list (splitting on spaces or other delimiters)
                        table_row = line.split()
                        all_rows.append(table_row)
                    else:
                        print(f"Skipping line: {line}")  # Optional: print skipped lines
            else:
                print(f"String '{target_string}' not found on page {page_number}.")
        else:
            print(f"No text found on page {page_number}.")

# Convert all_rows to a Pandas DataFrame
df = pd.DataFrame(all_rows)
df_dropped = df.drop(columns=[3,4,5,6,7,8])
df_dropped[df_dropped.columns[0]] = 'A' + df_dropped[df_dropped.columns[0]].astype(str)
df_dropped.columns = ['Name', 'RA', 'DEC']

print(df_dropped)
df_dropped['Name'] = df_dropped['Name'].astype(str)
# df_dropped['RA'] = df_dropped['RA'].astype(float)
# df_dropped['DEC'] = df_dropped['DEC'].astype(float)

# df_dropped['RA'] = pd.to_numeric(df_dropped['RA'], errors='coerce')
df_dropped['RA'] = df_dropped['RA'].astype(float)

df_dropped['DEC'] = df_dropped['DEC'].str.replace("−", "-", regex=True).str.strip()
df_dropped['DEC'] = pd.to_numeric(df_dropped['DEC'], errors='coerce')


# Optionally save the result to a CSV file
# df_dropped.to_csv("bcg_coords_table.csv", index=False)

# Print the resulting DataFrame
# print(df)
print(df_dropped)
print(df_dropped.dtypes)

## BCG LOCATION ##

In [ ]:
# Need to start using WCS I think...
r_wcs = WCS(coadd_r_hdul[0].header)
print(r_wcs)

bcg_coords_df = pd.read_csv("bcg_coords_table.csv")
A85_bcg = bcg_coords_df[bcg_coords_df['Name'] == cln]
A85_bcg.dtypes
A85_bcg_RA = A85_bcg['RA'].iloc[0]
A85_bcg_DEC = A85_bcg['DEC'].iloc[0]
print(A85_bcg_RA)
print(A85_bcg_DEC)

x_bcg, y_bcg = r_wcs.world_to_pixel_values(A85_bcg_RA, A85_bcg_DEC)
print(f"RA, DEC ({A85_bcg_RA}, {A85_bcg_DEC}) -> x: {x_bcg}, y: {y_bcg}")

# Display the fits

# Using the display function made above. Takes data array, minimum, maximum, and label. Not sure why multiplied by 255 though

coadd_r_scaled = display(coadd_r_data, -0.1, 250, 'fig1')*255


# Creating a smaller cutout to work with

# The array now has shape (2700,2700)
# coadd_r_small = coadd_r_data[2400:5100,2700:5400]
# coadd_r_small = coadd_r_data[3600:3900,3900:4200] #Smallest cutout

# small_coadd_r_scaled = display(coadd_r_small, -0.1, 250, 'fig1')*255

# brightest_pos = np.unravel_index(np.argmax(small_coadd_r_scaled[small_coadd_r_scaled <= 255], axis=None), small_coadd_r_scaled.shape)



plt.figure()
plt.imshow(coadd_r_scaled)
# plt.plot(brightest_pos[1],brightest_pos[0], 'ro', mfc = 'none')
plt.plot(x_bcg,y_bcg, 'ro', mfc = 'none')
#Inverting y since the image is "upside down"
plt.gca().invert_yaxis()
plt.xlabel('x (pixels)')
plt.ylabel('y (pixels)')
plt.xlim([3750,4250])
plt.ylim([3600,4000])
plt.title('Raw r Coadd skycorr_test')
plt.show()

# print(brightest_pos)
# print(small_coadd_r_scaled[brightest_pos[0]][brightest_pos[1]])
# print(coadd_r_data.shape)

# plt.figure()
# plt.imshow(data_mask)
# #Inverting y since the image is "upside down"
# plt.gca().invert_yaxis()
# plt.colorbar(label="Mask value", orientation="vertical") 
# plt.xlabel('x (pixels)')
# plt.ylabel('y (pixels)')
# plt.title('Segmentation sextractor output skycorr_test')
# plt.show()

BCG catalog worked!!! The coords appear to fall on the BCG. Now let's use sextractor to see if it picks the right mask..

In [ ]:
print(mask_r_data[int(y_bcg), int(x_bcg)])
print(mask_r_data)

In [ ]:
# Apply masks to all sources except the BCG
# bcg_unmsk_g_data = mask_single_coadd(coadd_g_data, mask_g_data, 13946)
bcg_unmsk_r_data = mask_single_coadd(coadd_r_data, mask_r_data, mask_r_data[int(y_bcg), int(x_bcg)])
# bcg_unmsk_i_data = mask_single_coadd(coadd_i_data, mask_i_data, 16265)


plt.figure()
plt.imshow(display(bcg_unmsk_r_data, -0.1, 250, 'fig1')*255)
#Inverting y since the image is "upside down"
plt.gca().invert_yaxis()
plt.colorbar(label="Intensity", orientation="vertical") 
plt.xlabel('x (pixels)')
plt.ylabel('y (pixels)')
plt.title('r Coadd BCG unmasked skycorr_test')
plt.show()

BCG has been properly unmasked. Now the background value estimation.

POSSIBLE ISSUE. I manually input 45 deg inclination below. How to make this automatic??

In [ ]:
log_spacings = np.logspace(1,3.6,15) #This includes BCG/ICL
sums_r, areas_r = fit_ellip_annuli('r', bcg_unmsk_r_data, np.array([int(x_bcg), int(y_bcg)]), log_spacings, 45)

In [ ]:
means_r_bkg = [a/b for a,b in zip(sums_r,areas_r)]
print(np.mean(means_r_bkg[-10]))

plt.figure(figsize=(6,4))

plt.scatter(log_spacings[1:], means_r_bkg, s = 50, color= 'r', marker ='o', label='r')
plt.plot(log_spacings[1:], means_r_bkg, 'r')

# plt.gca().invert_yaxis()
plt.xscale('log')
# plt.yscale('log')
plt.legend()
plt.xlabel('SMA (pix)')
plt.ylabel('Mean Intensity (Counts/pix)')
plt.title('Background Intensities')
plt.show()
